In [40]:
pip install --upgrade --quiet google-cloud-aiplatform[agent_engines,adk] googlemaps requests

In [41]:
import os
import json
import requests
from typing import Dict, Any, Optional, Sequence
import logging
import asyncio

# Import Google ADK and Vertex AI Reasoning Engine
import vertexai
from google.adk.agents import Agent, SequentialAgent
from google.adk.tools import google_search
from vertexai.agent_engines import AdkApp
from vertexai.preview import reasoning_engines
from vertexai.preview.generative_models import GenerativeModel, HarmCategory, HarmBlockThreshold, SafetySetting, GenerationResponse

In [42]:
# --- Configuration ---
PROJECT_ID: str = os.getenv("GOOGLE_CLOUD_PROJECT_ID", "qwiklabs-gcp-04-70cfc463a7f7")
LOCATION: str = os.getenv("GOOGLE_CLOUD_REGION", "us-central1") # e.g., "us-central1"
GOOGLE_MAPS_API_KEY: str = os.getenv("GOOGLE_MAPS_API_KEY", "")
NWS_USER_AGENT: str = "MyWeatherApp/1.0"
STAGING_BUCKET: str = os.getenv("STAGING_BUCKET", f"gs://{PROJECT_ID}-adk-staging")

# Initialize Vertex AI
vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)
print(f"Vertex AI Initialized for Project: {PROJECT_ID}, Location: {LOCATION}")

# --- Set up basic logging for callbacks ---
# Using basicConfig for consistent logging, but relying on print() for critical debug info in callbacks
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

Vertex AI Initialized for Project: qwiklabs-gcp-04-70cfc463a7f7, Location: us-central1


In [43]:
def _extract_location_from_prompt(prompt: str) -> Optional[str]:
    prompt_lower = prompt.lower()
    keywords = ["weather in", "forecast for", "temperature in", "how's the weather in"]
    for keyword in keywords:
        if keyword in prompt_lower:
            parts = prompt_lower.split(keyword, 1)
            if len(parts) > 1:
                return parts[1].strip().split('?')[0].split('.')[0].strip()
    return None

def _is_location_in_united_states(place_name: str) -> bool:
    if not GOOGLE_MAPS_API_KEY or GOOGLE_MAPS_API_KEY == "YOUR_GOOGLE_MAPS_API_KEY":
        return True # Default allow if key missing
    geocode_url: str = "https://maps.googleapis.com/maps/api/geocode/json"
    params: Dict[str, str] = {"address": place_name, "key": GOOGLE_MAPS_API_KEY}
    try:
        response = requests.get(geocode_url, params=params, timeout=5)
        data = response.json()
        if data["status"] == "OK" and data["results"]:
            for component in data["results"][0]["address_components"]:
                if "country" in component["types"] and component["short_name"] == "US":
                    return True
        return False
    except Exception:
        return True

def _check_for_malicious_input(prompt: str) -> bool:
    safety_model = GenerativeModel("gemini-2.5-fast-lite")
    safety_settings = [
        SafetySetting(category=HarmCategory.HARM_CATEGORY_HARASSMENT, threshold=HarmBlockThreshold.BLOCK_ONLY_HIGH),
        SafetySetting(category=HarmCategory.HARM_CATEGORY_HATE_SPEECH, threshold=HarmBlockThreshold.BLOCK_ONLY_HIGH),
        SafetySetting(category=HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE),
        SafetySetting(category=HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT, threshold=HarmBlockThreshold.BLOCK_ONLY_HIGH),
    ]
    try:
        response = safety_model.generate_content(prompt, safety_settings=safety_settings)
        if response.prompt_feedback and response.prompt_feedback.block_reason:
            return False
        return True
    except Exception:
        return True

def before_model_callback(query: str) -> str:
    logger.info(f"BEFORE_MODEL_CALLBACK - User Query: '{query}'")
    location_candidate = _extract_location_from_prompt(query)
    if location_candidate and not _is_location_in_united_states(location_candidate):
        raise ValueError(f"Location '{location_candidate}' is not in the United States.")
    if not _check_for_malicious_input(query):
        raise ValueError("Malicious input detected by Vertex AI Safety.")
    return query

def after_model_callback(query: str, model_response_text: str) -> None:
    logger.info(f"AFTER_MODEL_CALLBACK - Original Query: '{query}'")
    logger.info(f"AFTER_MODEL_CALLBACK - Model Response: '{model_response_text.strip()}'")

In [44]:
# --- Callback Functions ---

def before_model_callback(query: str) -> str:
    """
    Callback function executed before the model processes the user's prompt.
    1. Logs the user prompt.
    2. Ensures the user's location (if specified) is in the United States.
    3. Checks for malicious input using Vertex AI Safety Settings.
    """
    # Removed direct print statements, now using logger.info
    logger.info(f"BEFORE_MODEL_CALLBACK - User Query: '{query}'")

    # 2. Ensure the user's location is in the United States
    location_candidate = _extract_location_from_prompt(query)
    if location_candidate:
        logger.info(f"BEFORE_MODEL_CALLBACK - Extracted location candidate: '{location_candidate}'")
        if not _is_location_in_united_states(location_candidate):
            error_message = f"Location '{location_candidate}' is not in the United States. This agent only provides weather for US locations."
            logger.error(error_message)
            raise ValueError(error_message)
        else:
            logger.info(f"BEFORE_MODEL_CALLBACK - Location '{location_candidate}' confirmed to be in the United States.")
    else:
        logger.info("BEFORE_MODEL_CALLBACK - No specific location found in query for US check. Proceeding.")

    # 3. Check for malicious input using Vertex AI Safety
    if not _check_for_malicious_input(query):
        error_message = "Malicious input detected by Vertex AI Safety. Request blocked."
        logger.error(error_message) # Using logger.error for blocked requests
        raise ValueError(error_message)

    return query # Return the (potentially unchanged) query for model processing

def after_model_callback(query: str, model_response_text: str) -> None:
    """
    Callback function executed after the model generates a response.
    1. Logs the model response.
    """
    # Removed direct print statements, now using logger.info
    logger.info(f"AFTER_MODEL_CALLBACK - Original Query: '{query}'")
    logger.info(f"AFTER_MODEL_CALLBACK - Model Response: '{model_response_text.strip()}'")

In [45]:
# --- Tool Definitions (Python functions) ---

def get_lat_long_from_place(place_name: str) -> Optional[Dict[str, float]]:
    """
    Converts a human-readable place name into its latitude and longitude coordinates
    using the Google Maps Geocoding API.
    """
    if not GOOGLE_MAPS_API_KEY or GOOGLE_MAPS_API_KEY == "YOUR_GOOGLE_MAPS_API_KEY":
        logger.error("Google Maps API Key is not set. Cannot perform geocoding.")
        return None

    geocode_url: str = "https://maps.googleapis.com/maps/api/geocode/json"
    params: Dict[str, str] = {
        "address": place_name,
        "key": GOOGLE_MAPS_API_KEY
    }
    try:
        logger.debug(f"Calling Google Geocoding API for '{place_name}'...") # Replaced print with logger.debug
        response = requests.get(geocode_url, params=params, timeout=10)
        response.raise_for_status()
        data: Dict[str, Any] = response.json()
        logger.debug(f"Google Geocoding API response status: {data['status']}") # Replaced print with logger.debug

        if data["status"] == "OK" and data["results"]:
            location = data["results"][0]["geometry"]["location"]
            logger.debug(f"Found coordinates: Lat={location['lat']}, Long={location['lng']}") # Replaced print with logger.debug
            return {"latitude": location["lat"], "longitude": location["lng"]}
        else:
            logger.error(f"Geocoding failed for '{place_name}': {data.get('error_message', data['status'])}") # Replaced print with logger.error
            return None
    except requests.exceptions.RequestException as e:
        logger.error(f"Request error during geocoding API call: {e}") # Replaced print with logger.error
        return None
    except json.JSONDecodeError as e:
        logger.error(f"JSON decode error from geocoding API response: {e}") # Replaced print with logger.error
        return None
    except Exception as e:
        logger.error(f"UNEXPECTED ERROR in get_lat_long_from_place: {e}") # Replaced print with logger.error
        return None

def get_nws_weather_forecast(latitude: float, longitude: float) -> Optional[Dict[str, Any]]:
    """
    Retrieves the current weather forecast for a given latitude and longitude
    using the National Weather Service (NWS) API.
    """
    base_url: str = "https://api.weather.gov"
    points_url: str = f"{base_url}/points/{latitude},{longitude}"

    headers: Dict[str, str] = {"User-Agent": NWS_USER_AGENT}

    try:
        logger.debug(f"Calling NWS /points API for {latitude},{longitude}...") # Replaced print with logger.debug
        response_points = requests.get(points_url, headers=headers, timeout=10)
        response_points.raise_for_status()
        points_data: Dict[str, Any] = response_points.json()
        logger.debug(f"NWS /points API response: {points_data.get('properties', {}).get('forecast')}") # Replaced print with logger.debug

        forecast_url: Optional[str] = points_data["properties"].get("forecast")
        if not forecast_url:
            logger.error(f"Could not find forecast URL for {latitude},{longitude}.") # Replaced print with logger.error
            return None

        logger.debug(f"Calling NWS forecast API: {forecast_url}...") # Replaced print with logger.debug
        response_forecast = requests.get(forecast_url, headers=headers, timeout=10)
        response_forecast.raise_for_status()
        forecast_data: Dict[str, Any] = response_forecast.json()
        logger.debug(f"NWS forecast API response received (truncated): {str(forecast_data)[:200]}...") # Replaced print with logger.debug
        return forecast_data

    except requests.exceptions.RequestException as e:
        logger.error(f"Request error during NWS API call: {e}") # Replaced print with logger.error
        return None
    except json.JSONDecodeError as e:
        logger.error(f"JSON decode error from NWS API response: {e}") # Replaced print with logger.error
        return None
    except KeyError as e:
        logger.error(f"Missing key in NWS API response: {e}. Full response (truncated): {str(points_data)[:200]}...") # Replaced print with logger.error
        return None
    except Exception as e:
        logger.error(f"UNEXPECTED ERROR in get_nws_weather_forecast: {e}") # Replaced print with logger.error
        return None

def get_weather_summary(place_name: str) -> str:
    """
    Provides a weather summary or alert for a given place name.
    This function internally uses get_lat_long_from_place and get_nws_weather_forecast.

    Args:
        place_name: The name of the location for which to get the weather.

    Returns:
        A string containing the weather summary or an error message.
    """
    location_data: Optional[Dict[str, float]] = get_lat_long_from_place(place_name)
    if not location_data:
        return f"Could not find coordinates for {place_name}. Please try again with a different location."

    latitude: float = location_data["latitude"]
    # FIX: Changed 'lng' to 'longitude' to match the key returned by get_lat_long_from_place
    longitude: float = location_data["longitude"]

    weather_data: Optional[Dict[str, Any]] = get_nws_weather_forecast(latitude, longitude)
    if not weather_data:
        return f"Could not retrieve weather data for {place_name}."

    properties: Dict[str, Any] = weather_data.get("properties", {})
    forecast_periods: Optional[list] = properties.get("periods")

    if forecast_periods:
        current_or_next_period: Dict[str, Any] = forecast_periods[0] # Get the current or next forecast period
        name: str = current_or_next_period.get("name", "N/A")
        temperature: str = str(current_or_next_period.get("temperature", "N/A"))
        temperature_unit: str = current_or_next_period.get("temperatureUnit", "")
        forecast_text: str = current_or_next_period.get("detailedForecast", current_or_next_period.get("shortForecast", "No detailed forecast available."))

        summary: str = (
            f"**Weather for {place_name} ({name} Period):**\n"
            f"- Temperature: {temperature}°{temperature_unit}\n"
            f"- Conditions: {forecast_text}\n"
        )
        return summary
    else:
        return f"No forecast periods found for {place_name}."

In [48]:
# --- DEPLOYABLE AGENT CLASS ---
# We wrap the entire agent logic in a class. The __init__ takes pickle-able config.
# set_up() builds the un-pickle-able objects (Agents, Apps) on the server.

class WeatherSearchEngine:
    def __init__(self, project_id: str, location: str, maps_api_key: str, nws_user_agent: str):
        self.project_id = project_id
        self.location = location
        self.maps_api_key = maps_api_key
        self.nws_user_agent = nws_user_agent

    def set_up(self):
        # Imports must be inside set_up or available in the environment
        import vertexai
        import requests
        import asyncio
        import nest_asyncio
        from google.adk.agents import Agent, SequentialAgent
        from google.adk.tools import google_search
        from vertexai.agent_engines import AdkApp

        # Apply the asyncio patch
        nest_asyncio.apply()

        # Re-initialize Vertex AI in the remote container
        vertexai.init(project=self.project_id, location=self.location)

        # --- 1. Define Tools (Nested to access self.config) ---

        def get_lat_long_from_place(place_name: str) -> Optional[Dict[str, float]]:
            if not self.maps_api_key: return None
            try:
                response = requests.get(
                    "https://maps.googleapis.com/maps/api/geocode/json",
                    params={"address": place_name, "key": self.maps_api_key}, timeout=10
                )
                data = response.json()
                if data["status"] == "OK" and data["results"]:
                    loc = data["results"][0]["geometry"]["location"]
                    return {"latitude": loc["lat"], "longitude": loc["lng"]}
                return None
            except Exception as e:
                print(f"Geocoding error: {e}")
                return None

        def get_nws_weather_forecast(latitude: float, longitude: float) -> Optional[Dict[str, Any]]:
            try:
                headers = {"User-Agent": self.nws_user_agent}
                resp1 = requests.get(f"https://api.weather.gov/points/{latitude},{longitude}", headers=headers, timeout=10)
                resp1.raise_for_status()
                forecast_url = resp1.json()["properties"].get("forecast")
                if not forecast_url: return None
                resp2 = requests.get(forecast_url, headers=headers, timeout=10)
                resp2.raise_for_status()
                return resp2.json()
            except Exception as e:
                print(f"NWS error: {e}")
                return None

        def get_weather_summary(place_name: str) -> str:
            loc = get_lat_long_from_place(place_name)
            if not loc: return f"Could not find coordinates for {place_name}."
            data = get_nws_weather_forecast(loc["latitude"], loc["longitude"])
            if not data: return f"Could not retrieve weather for {place_name}."

            # Simple summary extraction
            try:
                period = data["properties"]["periods"][0]
                return f"Weather for {place_name}: {period['temperature']}°{period['temperatureUnit']}, {period['detailedForecast']}"
            except Exception:
                return "Error parsing weather data."

        # --- 2. Create Sub-Agents & Apps ---

        # Greeter
        self.greeter_agent = Agent(
            model="gemini-2.0-flash",
            name="GreeterAgent",
            description="Greeter",
            instruction="Respond to greetings warmly. Do not use tools."
        )
        self.greeter_app = AdkApp(agent=self.greeter_agent)

        # Search / Critique / Refine Pipeline
        self.search_sub_agent = Agent(
            model="gemini-2.0-flash", name="SearchAgent", tools=[google_search],
            instruction="Use Google Search to find info. Pass raw results to next stage."
        )
        self.critique_sub_agent = Agent(
            model="gemini-2.0-flash", name="CritiqueAgent",
            instruction="Critique the search results for the user's query. Suggest improvements."
        )
        self.refine_sub_agent = Agent(
            model="gemini-2.0-flash", name="RefineAgent",
            instruction="Refine the answer based on search results and critique. Provide final answer."
        )

        self.pipeline_agent = SequentialAgent(
            name="SearchRefinePipeline",
            description="Sequential search pipeline",
            sub_agents=[self.search_sub_agent, self.critique_sub_agent, self.refine_sub_agent]
        )
        self.pipeline_app = AdkApp(agent=self.pipeline_agent)

        # --- 3. Define Delegation Tools (Async Wrappers) ---

        # We need to run the async apps from sync tools if the root agent calls them sync,
        # OR define them as async tools if the Root Agent supports it.
        # Since we're inside a class, we define them here.

        async def delegate_to_greeter(user_input: str) -> str:
            print(f"HANDOFF: Greeter ({user_input})")
            resp = ""
            async for event in self.greeter_app.async_stream_query(user_id="del", message=user_input):
                c = event.get('content', {})
                if c and c.get('parts'):
                    for p in c['parts']:
                        if 'text' in p: resp += p['text']
            return resp

        async def sequential_search_workflow(user_query: str) -> str:
            print(f"HANDOFF: Pipeline ({user_query})")
            resp = ""
            async for event in self.pipeline_app.async_stream_query(user_id="del", message=user_query):
                c = event.get('content', {})
                if c and c.get('parts'):
                    for p in c['parts']:
                        if 'text' in p: resp += p['text']
            return resp

        # --- 4. Root Agent ---

        self.al_agent = Agent(
            model="gemini-2.0-flash",
            name="Al",
            instruction=(
                "You are Al. Delegate 'weather' questions to the 'get_weather_summary' tool directly. "
                "Delegate 'greetings' to 'delegate_to_greeter'. "
                "Delegate general knowledge/search questions to 'sequential_search_workflow'. "
            ),
            tools=[get_weather_summary, delegate_to_greeter, sequential_search_workflow]
        )
        self.root_app = AdkApp(agent=self.al_agent)

    def query(self, message: str) -> str:
        """
        Synchronous query method required by ReasoningEngine.
        It bridges to the async root_app execution.
        """
        response_text = ""

        async def run_async():
            nonlocal response_text
            async for event in self.root_app.async_stream_query(user_id="remote_user", message=message):
                c = event.get('content', {})
                if c and c.get('parts'):
                    for p in c['parts']:
                        if 'text' in p: response_text += p['text']

        # Run the async loop
        asyncio.run(run_async())
        return response_text

In [49]:
# --- DEPLOYMENT SECTION ---
print("\n--- DEPLOYING TO VERTEX AI ---")

DEPLOYED_AGENT_NAME = "MyMultiAgentWeatherSearch"
deployed_resource = None

async def deploy_and_test() -> None:
    global deployed_resource
    try:
        # We instantiate the class locally to create the config object...
        local_agent = WeatherSearchEngine(
            project_id=PROJECT_ID,
            location=LOCATION,
            maps_api_key=GOOGLE_MAPS_API_KEY,
            nws_user_agent=NWS_USER_AGENT
        )

        # ...but we pass the OBJECT to create(). Vertex AI pickles this object (just config strings),
        # uploads it, and then calls set_up() on the cloud side.
        deployed_resource = reasoning_engines.ReasoningEngine.create(
            reasoning_engine=local_agent,
            display_name=DEPLOYED_AGENT_NAME,
            requirements=[
                "google-cloud-aiplatform[agent_engines,adk]",
                "googlemaps",
                "requests",
                "nest_asyncio"
            ],
        )
        print(f"Deployed! Resource: {deployed_resource.resource_name}")

        # --- TESTING ---
        print("\n--- TESTING DEPLOYED AGENT ---")
        tests = ["Hello Al!", "What is the capital of France?", "Weather in New York City"]

        for q in tests:
            print(f"\nUser: {q}")

            # Client-side Pre-check
            try:
                safe_q = before_model_callback(q)
            except ValueError as e:
                print(f"Blocked: {e}"); continue

            # Call Remote Agent
            try:
                # deployed_resource.query() calls the query() method we defined in the class
                resp = deployed_resource.query(message=safe_q)
                print(f"Al: {resp}")

                # Client-side Post-check
                after_model_callback(safe_q, str(resp))
            except Exception as e:
                print(f"Error: {e}")

    except Exception as e:
        print(f"Deployment Failed: {e}")

# Run
await deploy_and_test()

INFO:vertexai.reasoning_engines._reasoning_engines:Using bucket qwiklabs-gcp-04-70cfc463a7f7-adk-staging



--- DEPLOYING TO VERTEX AI ---


INFO:vertexai.reasoning_engines._reasoning_engines:Writing to gs://qwiklabs-gcp-04-70cfc463a7f7-adk-staging/reasoning_engine/reasoning_engine.pkl
INFO:vertexai.reasoning_engines._reasoning_engines:Writing to gs://qwiklabs-gcp-04-70cfc463a7f7-adk-staging/reasoning_engine/requirements.txt
INFO:vertexai.reasoning_engines._reasoning_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.reasoning_engines._reasoning_engines:Writing to gs://qwiklabs-gcp-04-70cfc463a7f7-adk-staging/reasoning_engine/dependencies.tar.gz
INFO:vertexai.reasoning_engines._reasoning_engines:Creating ReasoningEngine
INFO:vertexai.reasoning_engines._reasoning_engines:Create ReasoningEngine backing LRO: projects/59692225922/locations/us-central1/reasoningEngines/7289611459061874688/operations/4090539918000914432
INFO:vertexai.reasoning_engines._reasoning_engines:ReasoningEngine created. Resource name: projects/59692225922/locations/us-central1/reasoningEngines/7289611459061874688
INFO:vertexai.reasoning_en

Deployed! Resource: projects/59692225922/locations/us-central1/reasoningEngines/7289611459061874688

--- TESTING DEPLOYED AGENT ---

User: Hello Al!
Al: Hey there! How's it going? Always a pleasure to hear from you!


User: What is the capital of France?
Al: The capital of France is Paris.


User: Weather in New York City
Al: Weather for New York City: 44°F, Cloudy, with a high near 44. Northeast wind around 10 mph.

